# MED source-free transfer: Panasonic 25C -> CMU eVTOL core-seven

This notebook runs `scripts/benchmarks/run_sftnn_transfer_cmu.py`: the
paper-faithful MED method (Shen et al. 2023, Algorithm 1) transferred from
the Panasonic 18650PF 25C source domain to the CMU eVTOL target domain,
compared against a zero-shot (no-adaptation) baseline. It follows the same
conventions as `34_backbone_transfer_panasonic_to_cmu.ipynb`: authenticated
clone/pull from GitHub, Drive-persisted paths from
`configs/paths/colab_drive.yaml`, fine-grained per-(seed, cell, method)
checkpointing with `--resume`, and results read straight back off Drive
rather than trusted to survive only in this runtime.

MED's transfer stage is source-free: it never sees Panasonic data/labels or
real CMU labels. Pseudo-labels come once from the frozen source model's
averaged-head output on CMU's *unlabeled* train-split missions; only the
target feature extractor is trained. Real CMU `soc_dynamic` labels are used
only afterwards, to score the held-out test-split missions -- and
`soc_dynamic` is a development dynamic-capacity reference, not measured
ground truth, so every result row is `label_is_ground_truth=false` /
`formal_benchmark_eligible=false`, matching the rest of this repo's CMU
eVTOL benchmarks.

Every result row also carries a wall-clock timing breakdown (source
pretraining, pseudo-label construction, adaptation, evaluation, and which
device it ran on) -- see the Compute-time section near the end.

## 0. Clone/pull the repo (authenticated) and install

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check token permission')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('cwd:', Path.cwd())

## Paths (Drive-persisted input/output roots)

Same `configs/paths/colab_drive.yaml` the backbone-transfer notebook uses --
results land on Drive so they survive a Colab disconnect, and the source
code path (`--panasonic-data-root`) doesn't depend on the raw `.mat` files
being baked into the git checkout.

In [ ]:
import torch
from battery_benchmark.utils.config import load_yaml

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
PANASONIC_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
CMU_BASE_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1'
CMU_CONDITION_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4'
CMU_MANIFEST = REPO / 'data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1.csv'
RESULT_ROOT = Path(PATHS['output_root']) / 'benchmarks/panasonic_to_cmu_evtol/sftnn_med_transfer'

for path in (PANASONIC_ROOT, CMU_BASE_ROOT, CMU_CONDITION_ROOT, CMU_MANIFEST):
    assert path.exists(), f'Missing required input: {path}'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('panasonic root:', PANASONIC_ROOT)
print('results (saved to Drive, survives a disconnect):', RESULT_ROOT)

## Before the full sweep: a smoke test

Run this once after a fresh clone/pull to catch a broken import or a bad
path before committing GPU time to the full sweep. Tiny epoch counts, one
cell, both methods, writes to a throwaway local (non-Drive) output folder.
A loose `--pseudo-threshold` is used here on purpose: with only 2
pretraining epochs the source model's predictions are too noisy for the
paper-default threshold (0.08) to keep any pseudo-labels at all -- that's
expected at this epoch count, not a bug.

In [ ]:
smoke_root = Path('/content/sftnn_smoke_test')
subprocess.run(
    [
        sys.executable, '-u', str(REPO / 'scripts/benchmarks/run_sftnn_transfer_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(smoke_root),
        '--cells', 'VAH01',
        '--methods', 'zero_shot', 'med',
        '--seeds', '42',
        '--source-epochs', '2', '--transfer-epochs', '2',
        '--early-stopping-patience', '1000',
        '--pseudo-threshold', '1.0',
        '--no-save-checkpoints',
        '--device', device,
    ],
    check=True,
)
print('smoke test passed')

## Setup for the full sweep

`SEEDS = [42]` by default. Defaults match the paper's own MED pretraining
regime (2000 source epochs) and the paper-default confidence threshold
(0.08).

In [ ]:
CELLS = ['VAH01', 'VAH17', 'VAH27', 'VAH30', 'VAH15', 'VAH12', 'VAH11']
SEEDS = [42]
SOURCE_EPOCHS, TRANSFER_EPOCHS, BATCH_SIZE = 2000, 200, 64
EARLY_STOPPING_PATIENCE = 30
PSEUDO_THRESHOLD = 0.08

def run_sftnn(methods: list[str]) -> None:
    command = [
        sys.executable, '-u', str(REPO / 'scripts/benchmarks/run_sftnn_transfer_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(RESULT_ROOT),
        '--cells', *CELLS,
        '--methods', *methods,
        '--seeds', *[str(s) for s in SEEDS],
        '--source-epochs', str(SOURCE_EPOCHS),
        '--transfer-epochs', str(TRANSFER_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--early-stopping-patience', str(EARLY_STOPPING_PATIENCE),
        '--pseudo-threshold', str(PSEUDO_THRESHOLD),
        '--device', device,
        '--resume',
    ]
    subprocess.run(command, check=True)

## Run: zero-shot + MED

With `--resume` and the fine-grained per-(seed, cell, method) checkpointing
in the script, re-running this cell after a disconnect skips everything
already finished.

In [ ]:
run_sftnn(['zero_shot', 'med'])

## Optional: DANN+MMD comparison baseline

Split into its own cell -- like the backbone-transfer notebook splits one
cell per backbone -- so a disconnect here doesn't require re-touching the
zero-shot/MED results above. Roughly doubles total runtime versus the cell
above, since it pretrains and adapts a second model. Skip this cell if you
only want the MED vs. zero-shot comparison.

In [ ]:
run_sftnn(['dann_mmd'])

## Results -- read straight from Drive

Reads every per-(seed, cell, method) result JSON directly off Drive under
`RESULT_ROOT`, the same way `34_backbone_transfer_panasonic_to_cmu.ipynb`
avoids depending on `cell_metrics.csv` staying fresh -- nothing here
requires re-cloning or re-running the training script, and it's safe to
re-run this section while a sweep above is still going or after a partial
run.

For context (not a direct comparison -- this is the paper's own Panasonic
cross-temperature domain, not CMU eVTOL): Shen et al. (2023) Table VI
reports MED averaging 4.17% MAE / 6.00% RMSE vs. 13.9% / 17.8% for the
source-pretrained baseline and 17.6% / 16.5% for the DANN+MMD baseline in
their own unsupervised cross-temperature setting.

In [ ]:
import json as _json
import pandas as pd
from IPython.display import display

method_order = ['zero_shot', 'med', 'dann_mmd']

def load_sftnn_results(result_root: Path) -> pd.DataFrame:
    """Read every per-(seed, cell, method) result JSON directly out of Drive."""
    rows = []
    for result_path in sorted(result_root.glob('seed_*/*/*.json')):
        rows.append(_json.loads(result_path.read_text(encoding='utf-8')))
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=['method', 'cell_id', 'seed'], keep='last')
    return df

results = load_sftnn_results(RESULT_ROOT)
print('rows:', len(results))
results

In [ ]:
if not results.empty:
    pivot = results.pivot_table(
        index='cell_id', columns='method',
        values=['overall_mae_percent', 'overall_rmse_percent'],
    ).round(3)
    display(pivot)

    curve = (
        results.groupby('method')['overall_mae_percent']
        .mean().reindex([m for m in method_order if m in results['method'].unique()]).round(3)
    )
    display(curve)
    curve.plot(
        kind='bar', figsize=(6, 4), title='MED source-free transfer, CMU eVTOL core-seven\n(dev labels -- not formal-benchmark-eligible)',
        ylabel='Mean MAE across 7 cells (%)', xlabel='Method',
    )
else:
    print('No results yet under', RESULT_ROOT)

## Compute-time results

Every row already has a timing breakdown in seconds:

* `med_source_pretrain_seconds` / `med_source_pretrain_epochs` -- one-off
  per seed (same value repeated across every cell/method row for that
  seed, since pretraining happens once and is then reused).
* `dann_source_pretrain_seconds` / `dann_source_pretrain_epochs` -- only
  populated if the DANN+MMD cell above was run.
* `pseudo_label_seconds` -- time to build the pseudo-labeled target
  dataset (MED only; confidence filtering runs per CMU mission).
* `adapt_seconds` -- time spent actually training (MED's target extractor,
  or DANN+MMD's target extractor + discriminator).
* `eval_seconds` -- time to score the held-out test-split missions.
* `device` -- `cpu` or `cuda`; seconds are meaningless without knowing
  which one produced them, so always read this alongside the timings.

In [ ]:
timing_columns = [
    'cell_id', 'method', 'seed', 'device',
    'med_source_pretrain_seconds', 'med_source_pretrain_epochs',
    'dann_source_pretrain_seconds', 'dann_source_pretrain_epochs',
    'pseudo_label_seconds', 'adapt_seconds', 'eval_seconds',
]

if not results.empty:
    timing = results[timing_columns].copy()
    timing['row_seconds'] = timing[
        ['pseudo_label_seconds', 'adapt_seconds', 'eval_seconds']
    ].sum(axis=1, skipna=True)
    display(timing)

    pretrain_once = timing.drop_duplicates(subset=['seed', 'device'])[
        ['seed', 'device', 'med_source_pretrain_seconds', 'dann_source_pretrain_seconds']
    ]
    total_pretrain = pretrain_once[
        ['med_source_pretrain_seconds', 'dann_source_pretrain_seconds']
    ].sum(skipna=True).sum()
    total_transfer_eval = timing['row_seconds'].sum()
    print(f'total pretraining time across seeds: {total_pretrain:.1f}s')
    print(f'total pseudo-label + adapt + eval time across all rows: {total_transfer_eval:.1f}s')
    print(f'grand total (pretrain once per seed + every row): {total_pretrain + total_transfer_eval:.1f}s')
else:
    print('No results yet under', RESULT_ROOT)

In [ ]:
import matplotlib.pyplot as plt

if not results.empty:
    med_rows = results[results['method'] == 'med']
    if not med_rows.empty:
        ax = med_rows.set_index('cell_id')['adapt_seconds'].plot(
            kind='bar', figsize=(10, 4), color='steelblue'
        )
        ax.set_ylabel('MED adaptation time (s)')
        ax.set_title('Per-cell MED adaptation wall-clock time')
        plt.tight_layout()
        plt.show()
    else:
        print("No 'med' rows yet")
else:
    print('No results yet under', RESULT_ROOT)